In [1]:
import ansys.aedt.core
import os
import tempfile
import time
from ansys.aedt.core import Maxwell3d
from ansys.aedt.core import Maxwell2d

AEDT_VERSION = "2025.1"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [2]:
# aedtPath=r'F:\KDH\Thesis\JEET\e10\refModel\e10_UserRemesh_ANSYSEM_2D.aedt'
# aedtPath=r'F:\KDH\ANSYS_TRC\ALH\Maxw_GS_EN_ILT\M04\Workshop_files\WS4.1\Maxw_GS_2DmagTrans1.aedt'
aedtPath=r'C:\ANSYS_Motor-CAD\2025_1_1\Tutorials\Ansys_Maxwell_Lab\e10_example\MaxwellLabTutorial.aedt'

In [3]:
# AEDT 파일 Lock 체크 및 안전한 파일 열기
import os
import time
import psutil
from pathlib import Path

def check_file_lock(file_path, timeout=30):
    """
    AEDT 파일이 잠겨있는지 확인하고 안전하게 열 수 있는 상태인지 체크
    
    Parameters:
    - file_path: 체크할 AEDT 파일 경로
    - timeout: 최대 대기 시간 (초)
    
    Returns:
    - True: 파일을 안전하게 열 수 있음
    - False: 파일이 잠겨있음
    """
    file_path = Path(file_path)
    
    # 파일 존재 여부 확인
    if not file_path.exists():
        print(f"❌ 파일이 존재하지 않습니다: {file_path}")
        return False
    
    # Lock 파일들 확인 (.lock, .lck, .tmp 등)
    lock_extensions = ['.lock', '.lck', '.tmp']
    lock_files = []
    
    for ext in lock_extensions:
        lock_file = file_path.with_suffix(file_path.suffix + ext)
        if lock_file.exists():
            lock_files.append(lock_file)
    
    # AEDT 관련 프로세스가 파일을 사용 중인지 확인
    def is_file_in_use():
        try:
            # 파일을 독점 모드로 열어보기
            with open(file_path, 'r+b') as f:
                pass
            return False
        except (IOError, OSError, PermissionError):
            return True
    
    # AEDT 관련 프로세스 확인
    def check_aedt_processes():
        aedt_processes = []
        for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
            try:
                name = proc.info['name'].lower()
                if any(aedt_name in name for aedt_name in ['ansysedt', 'maxwell', 'hfss', 'q3d']):
                    cmdline = proc.info.get('cmdline', [])
                    if cmdline and str(file_path) in ' '.join(cmdline):
                        aedt_processes.append(proc.info)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
        return aedt_processes
    
    print(f"📁 파일 확인: {file_path.name}")
    
    # Lock 파일 존재 확인
    if lock_files:
        print(f"⚠️  Lock 파일 발견: {[f.name for f in lock_files]}")
    
    # AEDT 프로세스 확인
    aedt_procs = check_aedt_processes()
    if aedt_procs:
        print(f"⚠️  AEDT 프로세스가 파일을 사용 중: {len(aedt_procs)}개 프로세스")
    
    # 파일 사용 상태 확인
    start_time = time.time()
    while time.time() - start_time < timeout:
        if not is_file_in_use() and not lock_files and not aedt_procs:
            print("✅ 파일을 안전하게 열 수 있습니다.")
            return True
        
        if time.time() - start_time < timeout:
            print(f"⏳ 파일이 사용 중입니다. 대기 중... ({int(time.time() - start_time)}/{timeout}초)")
            time.sleep(2)
            
            # 상태 재확인
            lock_files = [f for f in lock_files if f.exists()]
            aedt_procs = check_aedt_processes()
    
    print(f"❌ {timeout}초 대기 후에도 파일이 잠겨있습니다.")
    return False

def find_existing_aedt_instance(file_path):
    """
    해당 파일을 이미 열고 있는 AEDT 인스턴스가 있는지 확인
    
    Parameters:
    - file_path: 확인할 AEDT 파일 경로
    
    Returns:
    - Maxwell2d 객체 또는 None
    """
    try:
        import ansys.aedt.core as pyaedt
        
        # 실행 중인 AEDT Desktop 세션들 확인
        desktops = pyaedt.sessions.desktop_sessions
        
        if not desktops:
            print("🔍 실행 중인 AEDT Desktop 세션이 없습니다.")
            return None
        
        print(f"🔍 {len(desktops)}개의 AEDT Desktop 세션을 확인합니다...")
        
        for session_id, desktop in desktops.items():
            try:
                # Desktop의 프로젝트들 확인
                projects = desktop.odesktop.GetProjects()
                
                for project_name in projects:
                    project = desktop.odesktop.GetActiveProject()
                    if project and project.GetName() == project_name:
                        project_path = project.GetPath()
                        
                        # 파일 경로 비교 (정규화해서 비교)
                        if os.path.normpath(project_path) == os.path.normpath(file_path):
                            print(f"✅ 기존 AEDT 인스턴스에서 파일이 이미 열려있습니다!")
                            print(f"   Session ID: {session_id}")
                            print(f"   Project: {project_name}")
                            
                            # 기존 인스턴스에 연결
                            m2d = Maxwell2d(
                                project=project_name,
                                version=AEDT_VERSION,
                                new_desktop=False,
                                non_graphical=NG_MODE,
                                desktop_session_id=session_id
                            )
                            return m2d
                            
            except Exception as e:
                print(f"⚠️ Session {session_id} 확인 중 오류: {e}")
                continue
        
        print("🔍 해당 파일을 열고 있는 기존 인스턴스를 찾지 못했습니다.")
        return None
        
    except Exception as e:
        print(f"⚠️ 기존 인스턴스 검색 중 오류: {e}")
        return None

def safe_open_aedt(file_path, max_retries=3, use_existing=True):
    """
    AEDT 파일을 안전하게 여는 함수
    
    Parameters:
    - file_path: AEDT 파일 경로
    - max_retries: 최대 재시도 횟수
    - use_existing: 기존 인스턴스 사용 여부
    
    Returns:
    - Maxwell2d 객체 또는 None
    """
    
    # 1. 기존 인스턴스 확인 (옵션이 활성화된 경우)
    if use_existing:
        print("=== 기존 AEDT 인스턴스 확인 ===")
        existing_instance = find_existing_aedt_instance(file_path)
        if existing_instance:
            return existing_instance
    
    # 2. 새 인스턴스 생성
    for attempt in range(max_retries):
        print(f"\n=== 새 인스턴스 생성 시도 {attempt + 1}/{max_retries} ===")
        
        if check_file_lock(file_path):
            try:
                print("🔄 Maxwell2d 인스턴스 생성 중...")
                m2d = Maxwell2d(
                    project=file_path,
                    version=AEDT_VERSION,
                    new_desktop=False,
                    non_graphical=NG_MODE,
                )
                print("✅ Maxwell2d 인스턴스 생성 성공!")
                return m2d
                
            except Exception as e:
                print(f"❌ Maxwell2d 생성 실패: {e}")
                if attempt < max_retries - 1:
                    print("⏳ 5초 후 재시도...")
                    time.sleep(5)
        else:
            if attempt < max_retries - 1:
                print("⏳ 10초 후 재시도...")
                time.sleep(10)
    
    print("❌ 모든 시도가 실패했습니다.")
    return None

print("파일 Lock 체크 함수들이 정의되었습니다.")

파일 Lock 체크 함수들이 정의되었습니다.


In [4]:
from ansys.aedt.core import Maxwell2d

print("=== AEDT 파일 안전하게 열기 (기존 인스턴스 우선 사용) ===")

# 새로운 안전한 방식 - 기존 인스턴스 우선 사용
m2d = safe_open_aedt(aedtPath, max_retries=3, use_existing=True)

if m2d is not None:
    print(f"\n✅ 성공적으로 열렸습니다!")
    print(f"  프로젝트: {m2d.project_name}")
    print(f"  디자인: {m2d.design_name}")
    print(f"  솔루션 타입: {m2d.solution_type}")
    print(f"  버전: {m2d.aedt_version_id}")
    
    # 추가 정보 표시
    try:
        print(f"  Desktop PID: {m2d.desktop_class.aedt_process_id}")
        print(f"  Working Directory: {m2d.working_directory}")
    except Exception as e:
        print(f"  추가 정보 확인 중 오류: {e}")
        
else:
    print("\n❌ 파일을 열 수 없습니다. 다음을 확인해주세요:")
    print("  1. 다른 AEDT 인스턴스에서 파일이 열려있는지 확인")
    print("  2. 파일 경로가 올바른지 확인")
    print("  3. 파일 권한 확인")
    print("  4. AEDT 버전 호환성 확인")
    
    # 기존 인스턴스 없이 재시도 옵션
    print("\n💡 기존 인스턴스 확인 없이 재시도하려면:")
    print("    m2d = safe_open_aedt(aedtPath, use_existing=False)")

=== AEDT 파일 안전하게 열기 (기존 인스턴스 우선 사용) ===
=== 기존 AEDT 인스턴스 확인 ===
⚠️ 기존 인스턴스 검색 중 오류: module 'ansys.aedt.core' has no attribute 'sessions'

=== 새 인스턴스 생성 시도 1/3 ===
📁 파일 확인: MaxwellLabTutorial.aedt
✅ 파일을 안전하게 열 수 있습니다.
🔄 Maxwell2d 인스턴스 생성 중...
PyAEDT INFO: Parsing C:\ANSYS_Motor-CAD\2025_1_1\Tutorials\Ansys_Maxwell_Lab\e10_example\MaxwellLabTutorial.aedt.
PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: File C:\ANSYS_Motor-CAD\2025_1_1\Tutorials\Ansys_Maxwell_Lab\e10_example\MaxwellLabTutorial.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: Log on file C:\Users\user\AppData\Local\Temp\pyaedt_user_af0bd581-fe5e-4866-ad46-f12433868ad1.log is enabled.
PyAEDT INFO: Log on AEDT is enabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAED

## PJT Info

### getter all

In [ ]:
dirofItemsM2d = vars(m2d)
for attr in dirofItemsM2d:
    try:
        typeofDirListM2d[attr] = type(getattr(m2d, attr))
    except Exception as e:
        typeofDirListM2d[attr] = str(e)

### dev

In [ ]:
typeofItems={}
for k, v in vars(m2d).items():
    typeofItems[k] = type(v)
    print(f"{k}: {type(v)}")

varsofItems={}
for k, v in vars(m2d):
    varsofItems[k] = type(v)
    print(f"{k}: {type(v)}")

In [ ]:
# 디자인 이름 목록 추출
design_names = m2d.design_list
print("Designs in the project:", design_names)

## geometry objects parsing

In [ ]:
import json

# AEDT 프로젝트 열기

# 객체 리스트 가져오기
ob3dlist = m2d.modeler.object_list


In [ ]:
def parse_objects_with_instance(obj_list):
    parsed_objects = []
    for obj in obj_list:
        obj_info = {
            "name": getattr(obj, "name", None),
            "material": getattr(obj, "material_name", None),
            "model": getattr(obj, "model", None),
            "volume": getattr(obj, "volume", None),
            "mass": getattr(obj, "mass", None),
            "color": getattr(obj, "color", None),
            "transparency": getattr(obj, "transparency", None),
            "bounding_box": getattr(obj, "bounding_box", None),
            "solve_inside": getattr(obj, "solve_inside", None),
            "coordinate_system": getattr(obj, "part_coordinate_system", None),
            "faces": [],
            "edges": [],
            "vertices": [],
            "object": obj  # 객체 자체 저장
        }

        # Faces
        if hasattr(obj, "faces"):
            for face in obj.faces:
                face_info = {
                    "id": getattr(face, "id", None),
                    "area": getattr(face, "area", None),
                    "center": getattr(face, "center", None),
                    "vertices": []
                }
                if hasattr(face, "vertices"):
                    for vertex in face.vertices:
                        face_info["vertices"].append({
                            "id": getattr(vertex, "id", None),
                            "position": getattr(vertex, "position", None)
                        })
                obj_info["faces"].append(face_info)

        # Edges
        if hasattr(obj, "edges"):
            for edge in obj.edges:
                edge_info = {
                    "id": getattr(edge, "id", None),
                    "length": getattr(edge, "length", None)
                }
                obj_info["edges"].append(edge_info)

        # Vertices (global)
        if hasattr(obj, "vertices"):
            for vertex in obj.vertices:
                obj_info["vertices"].append({
                    "id": getattr(vertex, "id", None),
                    "position": getattr(vertex, "position", None)
                })

        parsed_objects.append(obj_info)
    return parsed_objects

# 사용 예시
parsed_objects = parse_objects_with_instance(ob3dlist)


In [ ]:
len(parsed_objects)

In [ ]:
parsed_structure = parse_object(parsed_objects[49]['object'], max_depth=2)


In [ ]:
import json

def safe_str(obj):
    """객체를 JSON 직렬화 가능하게 문자열로 변환 (예외 대비)"""
    try:
        json.dumps(obj)
        return obj
    except:
        return str(obj)

def parse_object(obj, depth=0, max_depth=3):
    """객체 속성을 재귀적으로 파싱해서 dict로 변환"""
    if depth > max_depth:
        return f"<Max depth {max_depth} reached>"

    result = {}
    for attr in dir(obj):
        if attr.startswith("_"):
            continue
        try:
            value = getattr(obj, attr)
            if callable(value):
                continue

            # 기본 타입이면 그대로 저장
            if isinstance(value, (int, float, str, bool, list, dict, type(None))):
                result[attr] = safe_str(value)
            # 재귀적 구조로 들어갈 수 있는 클래스형 속성
            # elif hasattr(value, "__dict__") or isinstance(value, object):
            elif  isinstance(value, object):
                result[attr] = parse_object(value, depth + 1, max_depth)
            else:
                result[attr] = safe_str(value)

        except Exception as e:
            result[attr] = f"<Error: {str(e)}>"

    return result



# JSON 출력
# print(json_string)


## Excitation

In [ ]:
excitation_list=m2d.excitations

In [ ]:
for name, obj in m2d.boundaries.items():
    if name in excitation_list:
        print(f"--- {name} ---")
        print(obj.props)  # 속성 딕셔너리 출력


### dev

In [ ]:
for name in m2d.excitations:
    if name in m2d.boundaries:
        bdry = m2d.boundaries[name]
        print(f"[{name}] Type: {bdry.type}")
        for key, value in bdry.props.items():
            print(f"    {key}: {value}")


In [ ]:
json_string = json.dumps(parsed_structure, indent=2, ensure_ascii=False)
print(json_string)


# 구조화된 클래스 데이터 직렬화 방법들

사용자 정의 클래스를 계층 구조를 유지하면서 기본 Python 데이터 타입이나 파일로 변환하는 방법들

In [5]:
# 1. 가장 일반적인 방법들
import json
import pickle
import yaml  # pip install pyyaml
from dataclasses import dataclass, asdict
import pandas as pd

print("=== 주요 직렬화 방법들 ===")

# 1) __dict__ 속성 사용 (기본 방법)
def to_dict_basic(obj):
    """기본적인 __dict__ 사용"""
    if hasattr(obj, '__dict__'):
        return obj.__dict__
    return obj

# 2) vars() 함수 사용 (동일한 결과)
def to_dict_vars(obj):
    """vars() 함수 사용"""
    try:
        return vars(obj)
    except TypeError:
        return obj

# 3) dataclass의 asdict() 사용 (dataclass용)
@dataclass
class ExampleDataClass:
    name: str
    value: int
    nested: dict = None

example_dc = ExampleDataClass("test", 42, {"sub": "value"})
print("dataclass asdict():", asdict(example_dc))

# 4) 재귀적 변환 함수 (복잡한 객체용)
def to_dict_recursive(obj, max_depth=5, current_depth=0):
    """재귀적으로 객체를 dict로 변환"""
    if current_depth >= max_depth:
        return str(obj)
    
    if hasattr(obj, '__dict__'):
        result = {}
        for key, value in obj.__dict__.items():
            if isinstance(value, (str, int, float, bool, type(None))):
                result[key] = value
            elif isinstance(value, (list, tuple)):
                result[key] = [to_dict_recursive(item, max_depth, current_depth + 1) for item in value]
            elif isinstance(value, dict):
                result[key] = {k: to_dict_recursive(v, max_depth, current_depth + 1) for k, v in value.items()}
            else:
                result[key] = to_dict_recursive(value, max_depth, current_depth + 1)
        return result
    elif isinstance(obj, (list, tuple)):
        return [to_dict_recursive(item, max_depth, current_depth + 1) for item in obj]
    elif isinstance(obj, dict):
        return {k: to_dict_recursive(v, max_depth, current_depth + 1) for k, v in obj.items()}
    else:
        return str(obj)

print("재귀적 변환 함수 정의 완료")

=== 주요 직렬화 방법들 ===
dataclass asdict(): {'name': 'test', 'value': 42, 'nested': {'sub': 'value'}}
재귀적 변환 함수 정의 완료


In [8]:
# 2. 파일 저장 방법들
print("\n=== 파일 저장 방법들 ===")

def save_to_json(obj, filename, use_custom_converter=True):
    """JSON 파일로 저장"""
    if use_custom_converter:
        data = to_dict_recursive(obj)
    else:
        data = obj
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)
    print(f"✓ JSON 저장: {filename}")

def save_to_pickle(obj, filename):
    """Pickle 파일로 저장 (객체 완전 보존)"""
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)
    print(f"✓ Pickle 저장: {filename}")

def save_to_yaml(obj, filename):
    """YAML 파일로 저장"""
    data = to_dict_recursive(obj)
    with open(filename, 'w', encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True)
    print(f"✓ YAML 저장: {filename}")

def save_to_csv_flat(obj, filename):
    """평면화된 CSV로 저장 (pandas 사용)"""
    data = to_dict_recursive(obj)
    
    # 중첩된 딕셔너리를 평면화
    def flatten_dict(d, parent_key='', sep='_'):
        items = []
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.extend(flatten_dict(v, new_key, sep=sep).items())
            elif isinstance(v, list):
                for i, item in enumerate(v):
                    if isinstance(item, dict):
                        items.extend(flatten_dict(item, f"{new_key}_{i}", sep=sep).items())
                    else:
                        items.append((f"{new_key}_{i}", item))
            else:
                items.append((new_key, v))
        return dict(items)
    
    flat_data = flatten_dict(data)
    df = pd.DataFrame([flat_data])
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"✓ CSV 저장: {filename}")

print("파일 저장 함수들 정의 완료")


=== 파일 저장 방법들 ===
파일 저장 함수들 정의 완료


In [6]:
# 3. AEDT 객체에 특화된 직렬화 함수
print("\n=== AEDT 전용 직렬화 함수 ===")

def serialize_aedt_object(aedt_obj, include_geometry=True, include_boundaries=True, max_depth=3):
    """AEDT 객체를 구조화된 딕셔너리로 변환"""
    result = {
        "project_info": {
            "project_name": getattr(aedt_obj, 'project_name', None),
            "design_name": getattr(aedt_obj, 'design_name', None),
            "design_type": getattr(aedt_obj, 'design_type', None),
            "solution_type": getattr(aedt_obj, 'solution_type', None),
            "version": getattr(aedt_obj, 'aedt_version_id', None),
        },
        "settings": {
            "model_units": getattr(aedt_obj.modeler, 'model_units', None) if hasattr(aedt_obj, 'modeler') else None,
            "working_directory": getattr(aedt_obj, 'working_directory', None),
        }
    }
    
    if include_geometry and hasattr(aedt_obj, 'modeler'):
        try:
            result["geometry"] = {
                "object_names": getattr(aedt_obj.modeler, 'object_names', []),
                "object_count": len(getattr(aedt_obj.modeler, 'object_names', [])),
                "objects_detail": []
            }
            
            # 각 객체의 상세 정보
            for obj in getattr(aedt_obj.modeler, 'object_list', []):
                obj_detail = {
                    "name": getattr(obj, 'name', None),
                    "material": getattr(obj, 'material_name', None),
                    "volume": getattr(obj, 'volume', None),
                    "mass": getattr(obj, 'mass', None),
                    "bounding_box": getattr(obj, 'bounding_box', None),
                    "color": getattr(obj, 'color', None),
                }
                result["geometry"]["objects_detail"].append(obj_detail)
        except Exception as e:
            result["geometry"] = {"error": str(e)}
    
    if include_boundaries and hasattr(aedt_obj, 'boundaries'):
        try:
            result["boundaries"] = {}
            for name, boundary in aedt_obj.boundaries.items():
                result["boundaries"][name] = {
                    "type": getattr(boundary, 'type', None),
                    "props": getattr(boundary, 'props', {}),
                }
        except Exception as e:
            result["boundaries"] = {"error": str(e)}
    
    # Excitations
    if hasattr(aedt_obj, 'excitations'):
        try:
            result["excitations"] = list(getattr(aedt_obj, 'excitations', []))
        except Exception as e:
            result["excitations"] = {"error": str(e)}
    
    return result

# m2d 객체에 적용 예제
print("AEDT 객체 직렬화 함수 정의 완료")
print("사용 예: serialized_m2d = serialize_aedt_object(m2d)")


=== AEDT 전용 직렬화 함수 ===
AEDT 객체 직렬화 함수 정의 완료
사용 예: serialized_m2d = serialize_aedt_object(m2d)


In [9]:
# 4. 실제 실행 - m2d 객체를 파일로 내보내기
print("\n=== m2d 객체 직렬화 및 파일 저장 ===")

# m2d 객체를 직렬화
try:
    serialized_m2d = serialize_aedt_object(m2d)
    print("✓ m2d 객체 직렬화 성공")
    
    # 구조 확인
    print(f"프로젝트명: {serialized_m2d['project_info']['project_name']}")
    print(f"디자인명: {serialized_m2d['project_info']['design_name']}")
    print(f"객체 개수: {serialized_m2d['geometry']['object_count']}")
    print(f"경계조건 개수: {len(serialized_m2d['boundaries'])}")
    print(f"여기조건 개수: {len(serialized_m2d['excitations'])}")
    
except Exception as e:
    print(f"직렬화 실패: {e}")
    serialized_m2d = None

# 파일로 저장
if serialized_m2d:
    try:
        # JSON으로 저장
        save_to_json(serialized_m2d, "m2d_maxwell_data.json", use_custom_converter=False)
        
        # YAML로도 저장
        save_to_yaml(serialized_m2d, "m2d_maxwell_data.yaml")
        
        # Pickle로도 저장 (완전한 객체 보존)
        save_to_pickle(serialized_m2d, "m2d_maxwell_data.pkl")
        
        # 평면화된 CSV로 저장
        save_to_csv_flat(serialized_m2d, "m2d_maxwell_data.csv")
        
        print("\n✓ 모든 형식으로 파일 저장 완료!")
        print("저장된 파일들:")
        print("  - m2d_maxwell_data.json (JSON 형식)")
        print("  - m2d_maxwell_data.yaml (YAML 형식)")
        print("  - m2d_maxwell_data.pkl (Pickle 형식)")
        print("  - m2d_maxwell_data.csv (평면화된 CSV)")
        
    except Exception as e:
        print(f"파일 저장 실패: {e}")


=== m2d 객체 직렬화 및 파일 저장 ===
✓ m2d 객체 직렬화 성공
프로젝트명: MaxwellLabTutorial
디자인명: Motor-CAD_tutorial
객체 개수: 51
경계조건 개수: 1
여기조건 개수: 43
✓ JSON 저장: m2d_maxwell_data.json
✓ YAML 저장: m2d_maxwell_data.yaml
✓ Pickle 저장: m2d_maxwell_data.pkl
✓ CSV 저장: m2d_maxwell_data.csv

✓ 모든 형식으로 파일 저장 완료!
저장된 파일들:
  - m2d_maxwell_data.json (JSON 형식)
  - m2d_maxwell_data.yaml (YAML 형식)
  - m2d_maxwell_data.pkl (Pickle 형식)
  - m2d_maxwell_data.csv (평면화된 CSV)


# YAML/JSON 파일 읽기 및 Maxwell 2D 설정 복원

YAML이나 JSON 파일에서 직렬화된 AEDT 데이터를 읽어와서 Maxwell 2D 인스턴스의 설정을 복원하는 함수들

## 5️⃣ Maxwell 2D 설정 복원 함수들

YAML/JSON/Pickle 파일에서 데이터를 로드하여 Maxwell 2D 프로젝트 설정을 복원하는 함수들입니다.

In [17]:
import yaml
import json
import pickle
from typing import Dict, Any, Optional

def load_data_from_file(file_path: str) -> Dict[str, Any]:
    """
    YAML, JSON, Pickle 파일에서 데이터를 로드
    
    Parameters:
    - file_path: 로드할 파일 경로
    
    Returns:
    - 로드된 데이터 딕셔너리
    """
    file_path = Path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"파일이 존재하지 않습니다: {file_path}")
    
    try:
        if file_path.suffix.lower() in ['.yaml', '.yml']:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = yaml.safe_load(f)
                print(f"✅ YAML 파일 로드 완료: {file_path}")
                
        elif file_path.suffix.lower() == '.json':
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                print(f"✅ JSON 파일 로드 완료: {file_path}")
                
        elif file_path.suffix.lower() in ['.pkl', '.pickle']:
            with open(file_path, 'rb') as f:
                data = pickle.load(f)
                print(f"✅ Pickle 파일 로드 완료: {file_path}")
                
        else:
            raise ValueError(f"지원하지 않는 파일 형식입니다: {file_path.suffix}")
        
        return data
        
    except Exception as e:
        print(f"❌ 파일 로드 실패: {e}")
        raise

# 테스트
print("📂 파일 로드 함수 정의 완료")

📂 파일 로드 함수 정의 완료


In [18]:
def restore_maxwell_2d_settings(m2d_instance, data: Dict[str, Any], 
                                verify_setup: bool = True) -> bool:
    """
    저장된 데이터에서 Maxwell 2D 설정을 복원
    
    Parameters:
    - m2d_instance: Maxwell 2D 인스턴스
    - data: 복원할 설정 데이터
    - verify_setup: 설정 검증 여부
    
    Returns:
    - 복원 성공 여부
    """
    try:
        print("🔄 Maxwell 2D 설정 복원 시작...")
        
        # 1. 기본 모델 정보 검증 및 설정
        if 'model_units' in data:
            current_units = m2d_instance.modeler.model_units
            target_units = data['model_units']
            
            if current_units != target_units:
                print(f"📏 모델 단위 변경: {current_units} → {target_units}")
                m2d_instance.modeler.model_units = target_units
            else:
                print(f"📏 모델 단위 확인: {current_units}")
        
        # 2. 솔루션 타입 설정
        if 'solution_type' in data:
            target_solution = data['solution_type']
            print(f"⚙️ 솔루션 타입: {target_solution}")
            # Note: 솔루션 타입은 프로젝트 생성 시 설정되므로 변경이 제한적임
        
        # 3. Geometry 복원 (기본 정보만)
        if 'geometry' in data and 'objects' in data['geometry']:
            geometry_objects = data['geometry']['objects']
            print(f"📐 Geometry 객체 수: {len(geometry_objects)}")
            
            # 기존 객체와 비교
            current_objects = list(m2d_instance.modeler.object_names)
            print(f"   현재 객체: {current_objects}")
            
            # 복원 대상 객체 리스트
            target_objects = list(geometry_objects.keys())
            print(f"   복원 대상: {target_objects}")
        
        # 4. Materials 복원
        if 'materials' in data:
            materials_data = data['materials']
            print(f"🧱 Materials 수: {len(materials_data)}")
            
            for mat_name, mat_props in materials_data.items():
                if mat_name not in m2d_instance.materials.material_keys:
                    print(f"   ➕ 새 Material 추가 필요: {mat_name}")
                    # Note: Material 생성 코드는 복잡하므로 여기서는 정보만 출력
                else:
                    print(f"   ✅ Material 존재: {mat_name}")
        
        # 5. Boundaries 복원 (정보 확인)
        if 'boundaries' in data:
            boundaries_data = data['boundaries']
            print(f"🔗 Boundaries 수: {len(boundaries_data)}")
            
            current_boundaries = m2d_instance.boundaries
            for boundary_name, boundary_props in boundaries_data.items():
                boundary_type = boundary_props.get('type', 'Unknown')
                print(f"   📋 {boundary_name}: {boundary_type}")
        
        # 6. Excitations 복원 (정보 확인)
        if 'excitations' in data:
            excitations_data = data['excitations']
            print(f"⚡ Excitations 수: {len(excitations_data)}")
            
            for exc_name, exc_props in excitations_data.items():
                exc_type = exc_props.get('type', 'Unknown')
                print(f"   🔌 {exc_name}: {exc_type}")
        
        # 7. Analysis 설정 복원 (정보 확인)
        if 'analysis' in data:
            analysis_data = data['analysis']
            print(f"📊 Analysis 설정:")
            
            if 'setups' in analysis_data:
                setups = analysis_data['setups']
                print(f"   Setup 수: {len(setups)}")
                
                for setup_name, setup_props in setups.items():
                    setup_type = setup_props.get('type', 'Unknown')
                    print(f"   🔧 {setup_name}: {setup_type}")
        
        # 8. 검증 수행
        if verify_setup:
            print("\n🔍 설정 검증 중...")
            validation_result = validate_maxwell_2d_setup(m2d_instance, data)
            
            if validation_result:
                print("✅ Maxwell 2D 설정 복원 및 검증 완료")
                return True
            else:
                print("⚠️ 설정 복원은 완료되었으나 일부 검증 실패")
                return False
        else:
            print("✅ Maxwell 2D 설정 복원 완료 (검증 스킵)")
            return True
            
    except Exception as e:
        print(f"❌ Maxwell 2D 설정 복원 실패: {e}")
        return False

print("🔧 Maxwell 2D 설정 복원 함수 정의 완료")

🔧 Maxwell 2D 설정 복원 함수 정의 완료


In [19]:
def validate_maxwell_2d_setup(m2d_instance, reference_data: Dict[str, Any]) -> bool:
    """
    Maxwell 2D 설정을 참조 데이터와 비교하여 검증
    
    Parameters:
    - m2d_instance: Maxwell 2D 인스턴스
    - reference_data: 참조할 설정 데이터
    
    Returns:
    - 검증 성공 여부
    """
    try:
        print("🔍 Maxwell 2D 설정 검증 시작...")
        validation_results = []
        
        # 1. 모델 단위 검증
        if 'model_units' in reference_data:
            current_units = m2d_instance.modeler.model_units
            expected_units = reference_data['model_units']
            
            units_match = current_units == expected_units
            validation_results.append(units_match)
            
            print(f"📏 모델 단위: {current_units} {'✅' if units_match else '❌'}")
            if not units_match:
                print(f"   예상: {expected_units}, 현재: {current_units}")
        
        # 2. Geometry 객체 수 검증
        if 'geometry' in reference_data and 'objects' in reference_data['geometry']:
            current_objects = list(m2d_instance.modeler.object_names)
            expected_objects = list(reference_data['geometry']['objects'].keys())
            
            objects_match = len(current_objects) == len(expected_objects)
            validation_results.append(objects_match)
            
            print(f"📐 Geometry 객체 수: {len(current_objects)} {'✅' if objects_match else '❌'}")
            if not objects_match:
                print(f"   예상: {len(expected_objects)}, 현재: {len(current_objects)}")
                print(f"   예상 객체: {expected_objects}")
                print(f"   현재 객체: {current_objects}")
        
        # 3. Materials 검증
        if 'materials' in reference_data:
            current_materials = set(m2d_instance.materials.material_keys)
            expected_materials = set(reference_data['materials'].keys())
            
            materials_match = expected_materials.issubset(current_materials)
            validation_results.append(materials_match)
            
            print(f"🧱 Materials: {'✅' if materials_match else '❌'}")
            if not materials_match:
                missing = expected_materials - current_materials
                print(f"   누락된 Materials: {missing}")
        
        # 4. Boundaries 검증
        if 'boundaries' in reference_data:
            current_boundaries = set(m2d_instance.boundaries)
            expected_boundaries = set(reference_data['boundaries'].keys())
            
            boundaries_match = expected_boundaries.issubset(current_boundaries)
            validation_results.append(boundaries_match)
            
            print(f"🔗 Boundaries: {'✅' if boundaries_match else '❌'}")
            if not boundaries_match:
                missing = expected_boundaries - current_boundaries
                print(f"   누락된 Boundaries: {missing}")
        
        # 5. Excitations 검증
        if 'excitations' in reference_data:
            try:
                current_excitations = set(m2d_instance.excitations)
                expected_excitations = set(reference_data['excitations'].keys())
                
                excitations_match = expected_excitations.issubset(current_excitations)
                validation_results.append(excitations_match)
                
                print(f"⚡ Excitations: {'✅' if excitations_match else '❌'}")
                if not excitations_match:
                    missing = expected_excitations - current_excitations
                    print(f"   누락된 Excitations: {missing}")
            except:
                print(f"⚡ Excitations: ⚠️ (검증 스킵)")
        
        # 6. Analysis Setup 검증
        if 'analysis' in reference_data and 'setups' in reference_data['analysis']:
            try:
                current_setups = list(m2d_instance.setups)
                expected_setups = list(reference_data['analysis']['setups'].keys())
                
                setups_match = len(current_setups) == len(expected_setups)
                validation_results.append(setups_match)
                
                print(f"📊 Analysis Setups: {'✅' if setups_match else '❌'}")
                if not setups_match:
                    print(f"   예상: {expected_setups}")
                    print(f"   현재: {current_setups}")
            except:
                print(f"📊 Analysis Setups: ⚠️ (검증 스킵)")
        
        # 전체 검증 결과
        overall_success = all(validation_results) if validation_results else True
        
        print(f"\n🎯 전체 검증 결과: {'✅ 성공' if overall_success else '❌ 실패'}")
        print(f"   검증 항목: {len(validation_results)}개")
        print(f"   성공한 항목: {sum(validation_results)}개")
        
        return overall_success
        
    except Exception as e:
        print(f"❌ 검증 중 오류 발생: {e}")
        return False

print("✅ Maxwell 2D 설정 검증 함수 정의 완료")

✅ Maxwell 2D 설정 검증 함수 정의 완료


In [20]:
def restore_maxwell_2d_from_file(m2d_instance, file_path: str, 
                                 verify_setup: bool = True) -> bool:
    """
    파일에서 Maxwell 2D 설정을 로드하고 복원하는 통합 함수
    
    Parameters:
    - m2d_instance: Maxwell 2D 인스턴스
    - file_path: YAML/JSON/Pickle 파일 경로
    - verify_setup: 설정 검증 여부
    
    Returns:
    - 복원 성공 여부
    """
    try:
        print(f"📁 파일에서 Maxwell 2D 설정 복원 시작: {file_path}")
        
        # 1. 파일에서 데이터 로드
        data = load_data_from_file(file_path)
        
        if not data:
            print("❌ 로드된 데이터가 비어있습니다.")
            return False
        
        # 2. Maxwell 2D 설정 복원
        success = restore_maxwell_2d_settings(m2d_instance, data, verify_setup)
        
        if success:
            print(f"✅ 파일에서 Maxwell 2D 설정 복원 완료: {Path(file_path).name}")
        else:
            print(f"❌ 파일에서 Maxwell 2D 설정 복원 실패: {Path(file_path).name}")
        
        return success
        
    except Exception as e:
        print(f"❌ 파일 복원 중 오류 발생: {e}")
        return False

def get_available_data_files(directory: str = ".", 
                           extensions: list = ['.yaml', '.yml', '.json', '.pkl', '.pickle']) -> list:
    """
    지정된 디렉토리에서 사용 가능한 데이터 파일 목록 반환
    
    Parameters:
    - directory: 검색할 디렉토리
    - extensions: 검색할 파일 확장자 리스트
    
    Returns:
    - 사용 가능한 파일 경로 리스트
    """
    directory = Path(directory)
    files = []
    
    for ext in extensions:
        files.extend(directory.glob(f"*{ext}"))
    
    files.sort()
    return [str(f) for f in files]

print("🔄 통합 복원 함수 정의 완료")

🔄 통합 복원 함수 정의 완료


## 6️⃣ Maxwell 2D 설정 복원 실습

앞서 저장한 YAML 파일을 읽어서 현재 Maxwell 2D 인스턴스 설정을 검증해보겠습니다.

In [21]:
# 현재 디렉토리에서 사용 가능한 데이터 파일 확인
current_dir = r"d:\KangDH\Emlab_emach\tools"
available_files = get_available_data_files(current_dir)

print("📂 사용 가능한 데이터 파일들:")
for i, file_path in enumerate(available_files, 1):
    file_name = Path(file_path).name
    file_size = Path(file_path).stat().st_size
    print(f"   {i}. {file_name} ({file_size:,} bytes)")

print(f"\n총 {len(available_files)}개 파일 발견")

📂 사용 가능한 데이터 파일들:
   1. m2d_data.json (23,033 bytes)
   2. m2d_maxwell_data.json (23,033 bytes)
   3. m2d_maxwell_data.pkl (8,952 bytes)
   4. m2d_maxwell_data.yaml (16,762 bytes)

총 4개 파일 발견


In [22]:
# YAML 파일에서 Maxwell 2D 설정 복원 및 검증
yaml_file_path = r"d:\KangDH\Emlab_emach\tools\m2d_maxwell_data.yaml"

if Path(yaml_file_path).exists():
    print(f"🎯 YAML 파일에서 Maxwell 2D 설정 복원 테스트")
    print(f"📁 파일: {Path(yaml_file_path).name}")
    
    # Maxwell 2D 인스턴스가 있는지 확인
    if 'm2d' in locals() and m2d is not None:
        print(f"✅ Maxwell 2D 인스턴스 확인: {m2d.project_name}")
        
        # 설정 복원 실행
        success = restore_maxwell_2d_from_file(m2d, yaml_file_path, verify_setup=True)
        
        if success:
            print("\n🎉 Maxwell 2D 설정 복원 및 검증 성공!")
        else:
            print("\n⚠️ Maxwell 2D 설정 복원 또는 검증에서 문제 발생")
            
    else:
        print("❌ Maxwell 2D 인스턴스가 없습니다. 먼저 Maxwell 2D를 열어주세요.")
        
else:
    print(f"❌ YAML 파일이 존재하지 않습니다: {yaml_file_path}")
    print("먼저 Maxwell 2D 데이터를 YAML로 저장해주세요.")

🎯 YAML 파일에서 Maxwell 2D 설정 복원 테스트
📁 파일: m2d_maxwell_data.yaml
✅ Maxwell 2D 인스턴스 확인: MaxwellLabTutorial
📁 파일에서 Maxwell 2D 설정 복원 시작: d:\KangDH\Emlab_emach\tools\m2d_maxwell_data.yaml
✅ YAML 파일 로드 완료: d:\KangDH\Emlab_emach\tools\m2d_maxwell_data.yaml
🔄 Maxwell 2D 설정 복원 시작...
🔗 Boundaries 수: 1
❌ Maxwell 2D 설정 복원 실패: 'str' object has no attribute 'get'
❌ 파일에서 Maxwell 2D 설정 복원 실패: m2d_maxwell_data.yaml

⚠️ Maxwell 2D 설정 복원 또는 검증에서 문제 발생
❌ Maxwell 2D 설정 복원 실패: 'str' object has no attribute 'get'
❌ 파일에서 Maxwell 2D 설정 복원 실패: m2d_maxwell_data.yaml

⚠️ Maxwell 2D 설정 복원 또는 검증에서 문제 발생


In [25]:
# 다양한 파일 형식에서 설정 복원 테스트
def test_restore_from_multiple_formats():
    """다양한 파일 형식에서 Maxwell 2D 설정 복원 테스트"""
    
    if 'm2d' not in locals() or m2d is None:
        print("❌ Maxwell 2D 인스턴스가 필요합니다.")
        return
    
    # 테스트할 파일들
    test_files = [
        r"d:\KangDH\Emlab_emach\tools\m2d_maxwell_data.yaml",
        r"d:\KangDH\Emlab_emach\tools\m2d_maxwell_data.json",
        r"d:\KangDH\Emlab_emach\tools\m2d_maxwell_data.pkl"
    ]
    
    print("🧪 다양한 파일 형식에서 Maxwell 2D 설정 복원 테스트\n")
    
    for file_path in test_files:
        if Path(file_path).exists():
            file_name = Path(file_path).name
            file_ext = Path(file_path).suffix
            
            print(f"📁 {file_ext.upper()} 파일 테스트: {file_name}")
            print("-" * 50)
            
            # 복원 실행 (검증은 빠르게 하기 위해 False로 설정)
            success = restore_maxwell_2d_from_file(m2d, file_path, verify_setup=False)
            
            if success:
                print(f"✅ {file_ext.upper()} 파일 복원 성공\n")
            else:
                print(f"❌ {file_ext.upper()} 파일 복원 실패\n")
        else:
            file_name = Path(file_path).name
            print(f"⚠️ 파일 없음: {file_name}\n")


In [30]:

# 테스트 실행
restore_maxwell_2d_from_file(m2d, "m2d_maxwell_data.yaml", verify_setup=True)
   

📁 파일에서 Maxwell 2D 설정 복원 시작: m2d_maxwell_data.yaml
✅ YAML 파일 로드 완료: m2d_maxwell_data.yaml
🔄 Maxwell 2D 설정 복원 시작...
🔗 Boundaries 수: 1
❌ Maxwell 2D 설정 복원 실패: 'str' object has no attribute 'get'
❌ 파일에서 Maxwell 2D 설정 복원 실패: m2d_maxwell_data.yaml


False

## 7️⃣ 사용법 가이드

### 📋 Maxwell 2D 설정 복원 워크플로우

1. **Maxwell 2D 인스턴스 생성**
   ```python
   m2d = Maxwell2d(project="MyProject", solution_type="Magnetostatic")
   ```

2. **설정 데이터 저장** (이전 섹션 참조)
   ```python
   serialized_data = serialize_maxwell_2d(m2d)
   export_to_yaml(serialized_data, "my_maxwell_setup.yaml")
   ```

3. **설정 복원**
   ```python
   # 단순 복원 (검증 없음)
   restore_maxwell_2d_from_file(m2d, "my_maxwell_setup.yaml", verify_setup=False)
   
   # 복원 + 검증
   restore_maxwell_2d_from_file(m2d, "my_maxwell_setup.yaml", verify_setup=True)
   ```

4. **개별 함수 사용**
   ```python
   # 데이터 로드만
   data = load_data_from_file("my_maxwell_setup.yaml")
   
   # 설정 복원만
   restore_maxwell_2d_settings(m2d, data, verify_setup=False)
   
   # 검증만
   validate_maxwell_2d_setup(m2d, data)
   ```

### ⚠️ 주의사항

- **Geometry 복원**: 현재 버전은 geometry 정보 확인만 하며, 실제 geometry 생성은 별도 구현 필요
- **Materials**: 기존 materials과 비교만 하며, 새로운 material 생성은 별도 구현 필요  
- **Boundaries/Excitations**: AEDT API 제약으로 일부 항목은 검증만 가능
- **Analysis Setup**: Setup 정보 확인만 하며, 상세 설정 복원은 추가 개발 필요

### 🔧 확장 가능한 부분

1. **완전한 Geometry 복원**: Modeler API를 활용한 geometry 재생성
2. **Material 자동 생성**: 저장된 material 속성을 바탕으로 새 material 생성
3. **Boundary/Excitation 복원**: 상세 설정을 포함한 완전한 복원
4. **Analysis Setup 복원**: Solver 설정, Mesh 설정 등 완전한 복원

In [ ]:
# 1. 파일에서 데이터 로드 함수들
import json
import yaml
import pickle
from pathlib import Path

def load_aedt_data(file_path):
    """
    YAML, JSON, 또는 Pickle 파일에서 AEDT 데이터를 로드
    
    Parameters:
    - file_path: 파일 경로 (str 또는 Path)
    
    Returns:
    - dict: 로드된 AEDT 데이터
    """
    file_path = Path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}")
    
    print(f"📂 파일 로드 중: {file_path.name}")
    
    try:
        if file_path.suffix.lower() in ['.yaml', '.yml']:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = yaml.safe_load(f)
            print("✅ YAML 파일 로드 성공")
            
        elif file_path.suffix.lower() == '.json':
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            print("✅ JSON 파일 로드 성공")
            
        elif file_path.suffix.lower() == '.pkl':
            with open(file_path, 'rb') as f:
                data = pickle.load(f)
            print("✅ Pickle 파일 로드 성공")
            
        else:
            raise ValueError(f"지원되지 않는 파일 형식: {file_path.suffix}")
        
        # 데이터 구조 확인
        if isinstance(data, dict):
            print(f"  프로젝트: {data.get('project_info', {}).get('project_name', 'Unknown')}")
            print(f"  디자인: {data.get('project_info', {}).get('design_name', 'Unknown')}")
            
            if 'geometry' in data:
                object_count = data['geometry'].get('object_count', 0)
                print(f"  객체 수: {object_count}")
            
            if 'boundaries' in data and isinstance(data['boundaries'], dict):
                boundary_count = len(data['boundaries'])
                print(f"  경계조건 수: {boundary_count}")
                
            if 'excitations' in data and isinstance(data['excitations'], list):
                excitation_count = len(data['excitations'])
                print(f"  여기조건 수: {excitation_count}")
        
        return data
        
    except Exception as e:
        print(f"❌ 파일 로드 실패: {e}")
        raise

print("데이터 로드 함수 정의 완료")

In [ ]:
# 2. Maxwell 2D 설정 복원 함수들

def apply_project_settings(m2d_instance, project_data):
    """
    프로젝트 기본 설정 적용
    
    Parameters:
    - m2d_instance: Maxwell2d 인스턴스
    - project_data: 로드된 프로젝트 데이터
    """
    try:
        settings = project_data.get('settings', {})
        
        # 모델 단위 설정
        if 'model_units' in settings and settings['model_units']:
            current_units = m2d_instance.modeler.model_units
            target_units = settings['model_units']
            
            if current_units != target_units:
                print(f"🔧 모델 단위 변경: {current_units} → {target_units}")
                m2d_instance.modeler.model_units = target_units
            else:
                print(f"✅ 모델 단위 일치: {current_units}")
        
        # 솔루션 타입 확인
        project_info = project_data.get('project_info', {})
        if 'solution_type' in project_info:
            current_solution = m2d_instance.solution_type
            target_solution = project_info['solution_type']
            print(f"📊 솔루션 타입: {current_solution} (목표: {target_solution})")
            
            if current_solution != target_solution:
                print(f"⚠️ 솔루션 타입이 다릅니다. 수동으로 변경이 필요할 수 있습니다.")
        
        print("✅ 프로젝트 설정 적용 완료")
        
    except Exception as e:
        print(f"❌ 프로젝트 설정 적용 실패: {e}")

def verify_geometry_objects(m2d_instance, geometry_data):
    """
    Geometry 객체들 검증
    
    Parameters:
    - m2d_instance: Maxwell2d 인스턴스
    - geometry_data: Geometry 데이터
    """
    try:
        if not geometry_data or 'objects_detail' not in geometry_data:
            print("⚠️ Geometry 데이터가 없습니다.")
            return
        
        current_objects = set(m2d_instance.modeler.object_names)
        target_objects_detail = geometry_data['objects_detail']
        target_objects = {obj['name'] for obj in target_objects_detail if obj.get('name')}
        
        print(f"📊 객체 비교:")
        print(f"  현재 모델: {len(current_objects)}개 객체")
        print(f"  목표 모델: {len(target_objects)}개 객체")
        
        # 일치하는 객체
        matching = current_objects & target_objects
        if matching:
            print(f"  ✅ 일치하는 객체: {len(matching)}개")
            
        # 누락된 객체
        missing = target_objects - current_objects
        if missing:
            print(f"  ❌ 누락된 객체: {len(missing)}개")
            for obj in list(missing)[:5]:  # 처음 5개만 표시
                print(f"    - {obj}")
            if len(missing) > 5:
                print(f"    ... 및 {len(missing) - 5}개 더")
        
        # 추가된 객체
        extra = current_objects - target_objects
        if extra:
            print(f"  ➕ 추가된 객체: {len(extra)}개")
            for obj in list(extra)[:5]:  # 처음 5개만 표시
                print(f"    - {obj}")
            if len(extra) > 5:
                print(f"    ... 및 {len(extra) - 5}개 더")
        
        # 재료 정보 비교 (일치하는 객체들만)
        print(f"\n🔍 재료 정보 검증:")
        material_mismatches = 0
        
        for obj_detail in target_objects_detail:
            obj_name = obj_detail.get('name')
            target_material = obj_detail.get('material')
            
            if obj_name in current_objects and target_material:
                try:
                    current_obj = m2d_instance.modeler[obj_name]
                    current_material = current_obj.material_name
                    
                    if current_material != target_material:
                        material_mismatches += 1
                        print(f"    ⚠️ {obj_name}: {current_material} ≠ {target_material}")
                except:
                    pass
        
        if material_mismatches == 0:
            print(f"    ✅ 모든 재료 정보가 일치합니다.")
        else:
            print(f"    ❌ {material_mismatches}개 객체의 재료가 다릅니다.")
            
    except Exception as e:
        print(f"❌ Geometry 검증 실패: {e}")

def verify_excitations(m2d_instance, excitation_data):
    """
    여기조건들 검증
    
    Parameters:
    - m2d_instance: Maxwell2d 인스턴스
    - excitation_data: 여기조건 데이터 (리스트)
    """
    try:
        if not excitation_data:
            print("⚠️ 여기조건 데이터가 없습니다.")
            return
        
        current_excitations = set(m2d_instance.excitations)
        target_excitations = set(excitation_data)
        
        print(f"📊 여기조건 비교:")
        print(f"  현재 모델: {len(current_excitations)}개")
        print(f"  목표 모델: {len(target_excitations)}개")
        
        # 일치하는 여기조건
        matching = current_excitations & target_excitations
        if matching:
            print(f"  ✅ 일치하는 여기조건: {len(matching)}개")
            
        # 누락된 여기조건
        missing = target_excitations - current_excitations
        if missing:
            print(f"  ❌ 누락된 여기조건: {len(missing)}개")
            for exc in list(missing)[:5]:
                print(f"    - {exc}")
            if len(missing) > 5:
                print(f"    ... 및 {len(missing) - 5}개 더")
        
        # 추가된 여기조건
        extra = current_excitations - target_excitations
        if extra:
            print(f"  ➕ 추가된 여기조건: {len(extra)}개")
            for exc in list(extra)[:5]:
                print(f"    - {exc}")
            if len(extra) > 5:
                print(f"    ... 및 {len(extra) - 5}개 더")
                
    except Exception as e:
        print(f"❌ 여기조건 검증 실패: {e}")

print("Maxwell 2D 설정 복원 함수들 정의 완료")